# Cycle 3 - 01 Data Cleaning and Recoding

This notebook prepares the cleaned data for the Cycle 3 project.

**Research question:** Is the proportion of current alcohol use different between male and female students?

**Main variables:**
- `WhatIsYourSex`: group variable, Female vs. Male
- `CurrentAlcoholUse`: response variable, recoded into 0/1

**Extension variable:**
- `InWhatGradeAreYou`: used later for grade-level descriptive comparison

This notebook creates two cleaned files:
1. `data/processed/cycle3_cleaned_main.csv` for the main two-proportion analysis
2. `data/processed/cycle3_cleaned_grade_extension.csv` for the grade-level extension


## 1. Import libraries and create folders

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# This notebook is designed to be run from the project root folder.
# If you run it inside the notebooks/ folder, the code below will still try to locate the project root.
current_dir = Path.cwd()

if (current_dir / "data" / "raw").exists():
    project_dir = current_dir
elif (current_dir.parent / "data" / "raw").exists():
    project_dir = current_dir.parent
else:
    project_dir = current_dir

print("Project directory:", project_dir)

# Create required folders
folders = [
    "data/raw",
    "data/processed",
    "outputs/figures",
    "outputs/tables",
    "outputs/summary",
    "references",
    "report"
]

for folder in folders:
    (project_dir / folder).mkdir(parents=True, exist_ok=True)

print("Folders checked / created.")

Project directory: d:\作業\統計學實習\cycle3\project-cycle-3
Folders checked / created.


## 2. Load raw data

Make sure the original file is placed here:

```text
data/raw/YRBS_2007.csv
```

In [8]:
raw_data_path = project_dir / "data" / "raw" / "YRBS_2007.csv"

if not raw_data_path.exists():
    raise FileNotFoundError(
        f"Cannot find {raw_data_path}. Please put YRBS_2007.csv in data/raw/."
    )

df_raw = pd.read_csv(raw_data_path)

print("Raw data shape:", df_raw.shape)
df_raw.head()

Raw data shape: (14041, 103)


,RaceEth,HowOldAreYou,WhatIsYourSex,InWhatGradeAreYou,AreYouHispanicOrLatino,WhatIsYourRace,HowTallAreYouWithoutShoesInMeters,HowMuchDoYouWeighWithoutShoesInKG,BicyleHelmetUse,SeatBeltUse,...,InjuredWhileExercising,HIVTesting,SunscreenUse,SunProtection,Sleep,HealthInGeneral,BMIPCT,weight,stratum,psu
0,7.0,4.0,2.0,2.0,1.0,C,NaN,NaN,2.0,1.0,...,3.0,2.0,1.0,1.0,5.0,3.0,NaN,1.5104,101,11030
1,5.0,7.0,2.0,2.0,2.0,E,1.70,68.04,4.0,4.0,...,2.0,3.0,1.0,5.0,4.0,3.0,66.531824,1.8559,101,11030
2,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,5.0,3.0,...,2.0,3.0,2.0,1.0,1.0,1.0,NaN,1.8559,101,11030
3,7.0,1.0,1.0,1.0,1.0,A,1.63,79.38,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,98.174319,1.3264,101,11030
4,7.0,1.0,1.0,5.0,1.0,B,NaN,NaN,6.0,5.0,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.3264,101,11030


## 3. Select variables

For the main analysis, we need sex and current alcohol use.

For the extension, we also keep grade level.

In [9]:
main_vars = ["WhatIsYourSex", "CurrentAlcoholUse"]
extension_vars = ["WhatIsYourSex", "CurrentAlcoholUse", "InWhatGradeAreYou"]

# Check whether all required columns exist
missing_cols = [col for col in extension_vars if col not in df_raw.columns]
if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

df_main_raw = df_raw[main_vars].copy()
df_ext_raw = df_raw[extension_vars].copy()

print("Selected main variables:")
display(df_main_raw.head())

print("Selected extension variables:")
display(df_ext_raw.head())

Selected main variables:


,WhatIsYourSex,CurrentAlcoholUse
0,2.0,NaN
1,2.0,NaN
2,2.0,NaN
3,1.0,1.0
4,1.0,NaN


Selected extension variables:


,WhatIsYourSex,CurrentAlcoholUse,InWhatGradeAreYou
0,2.0,NaN,2.0
1,2.0,NaN,2.0
2,2.0,NaN,NaN
3,1.0,1.0,1.0
4,1.0,NaN,5.0


## 4. Check original coding

Before cleaning, we check the raw value counts. This is useful for documenting missing or invalid codes.

In [10]:
for col in extension_vars:
    print("=" * 60)
    print(col)
    print(df_raw[col].value_counts(dropna=False).sort_index())

WhatIsYourSex
WhatIsYourSex
1.0    7036
2.0    6992
NaN      13
Name: count, dtype: int64
CurrentAlcoholUse
CurrentAlcoholUse
1.0    6946
2.0    2735
3.0    1369
4.0     839
5.0     555
6.0     105
7.0     120
NaN    1372
Name: count, dtype: int64
InWhatGradeAreYou
InWhatGradeAreYou
1.0    3467
2.0    3482
3.0    3480
4.0    3529
5.0      14
NaN      69
Name: count, dtype: int64


## 5. Clean and recode main analysis data

Main analysis keeps observations with valid sex and valid current alcohol use.

**Coding rules:**

`WhatIsYourSex`:
- `1 = Female`
- `2 = Male`

`CurrentAlcoholUse`:
- original code `1` means no current alcohol use → recode to `0`
- original codes `2–7` mean current alcohol use → recode to `1`


In [11]:
# Keep valid values only
df_main_clean = df_main_raw.dropna(subset=main_vars).copy()

df_main_clean = df_main_clean[
    df_main_clean["WhatIsYourSex"].isin([1, 2])
    & df_main_clean["CurrentAlcoholUse"].isin([1, 2, 3, 4, 5, 6, 7])
].copy()

# Recode sex
df_main_clean["sex_group"] = df_main_clean["WhatIsYourSex"].map({
    1: "Female",
    2: "Male"
})

# Recode current alcohol use
df_main_clean["current_alcohol_use"] = np.where(
    df_main_clean["CurrentAlcoholUse"] == 1,
    0,
    1
)

# Keep clean columns only
df_main_final = df_main_clean[["sex_group", "current_alcohol_use"]].copy()

print("Main cleaned data shape:", df_main_final.shape)
display(df_main_final.head())

Main cleaned data shape: (12659, 2)


,sex_group,current_alcohol_use
3,Female,0
5,Female,0
6,Female,0
7,Female,0
8,Female,0


## 6. Check cleaned main data

This table should match the numbers used in the main analysis later.

In [12]:
print("Sample size by sex:")
print(df_main_final["sex_group"].value_counts())

print("Current alcohol use table:")
main_crosstab = pd.crosstab(
    df_main_final["sex_group"],
    df_main_final["current_alcohol_use"],
    margins=True
)

display(main_crosstab)

print("Proportion of current alcohol use by sex:")
main_prop = df_main_final.groupby("sex_group")["current_alcohol_use"].mean()
display(main_prop)

Sample size by sex:
sex_group
Female    6425
Male      6234
Name: count, dtype: int64
Current alcohol use table:


current_alcohol_use,0,1,All
sex_group,,,
Female,3561,2864,6425
Male,3381,2853,6234
All,6942,5717,12659


Proportion of current alcohol use by sex:


sex_group
Female    0.445759
Male      0.457652
Name: current_alcohol_use, dtype: float64

## 7. Save cleaned main data

In [13]:
processed_dir = project_dir / "data" / "processed"
main_output_path = processed_dir / "cycle3_cleaned_main.csv"

df_main_final.to_csv(main_output_path, index=False)

print("Saved main cleaned data to:", main_output_path)

Saved main cleaned data to: d:\作業\統計學實習\cycle3\project-cycle-3\data\processed\cycle3_cleaned_main.csv


## 8. Clean and recode extension data

The extension will compare current alcohol use by sex **within each grade level**.

We keep grades 1–4 and map them as:
- `1 = 9th grade`
- `2 = 10th grade`
- `3 = 11th grade`
- `4 = 12th grade`

If code `5` appears, it is treated as outside the main 9th–12th grade comparison and removed for this extension.

In [14]:
df_ext_clean = df_ext_raw.dropna(subset=extension_vars).copy()

df_ext_clean = df_ext_clean[
    df_ext_clean["WhatIsYourSex"].isin([1, 2])
    & df_ext_clean["CurrentAlcoholUse"].isin([1, 2, 3, 4, 5, 6, 7])
    & df_ext_clean["InWhatGradeAreYou"].isin([1, 2, 3, 4])
].copy()

# Recode sex
df_ext_clean["sex_group"] = df_ext_clean["WhatIsYourSex"].map({
    1: "Female",
    2: "Male"
})

# Recode current alcohol use
df_ext_clean["current_alcohol_use"] = np.where(
    df_ext_clean["CurrentAlcoholUse"] == 1,
    0,
    1
)

# Recode grade level
df_ext_clean["grade_level"] = df_ext_clean["InWhatGradeAreYou"].map({
    1: "9th grade",
    2: "10th grade",
    3: "11th grade",
    4: "12th grade"
})

df_ext_final = df_ext_clean[["grade_level", "sex_group", "current_alcohol_use"]].copy()

print("Extension cleaned data shape:", df_ext_final.shape)
display(df_ext_final.head())

Extension cleaned data shape: (12600, 3)


,grade_level,sex_group,current_alcohol_use
3,9th grade,Female,0
5,11th grade,Female,0
6,11th grade,Female,0
7,11th grade,Female,0
8,11th grade,Female,0


## 9. Check cleaned extension data

In [15]:
print("Sample size by grade and sex:")

ext_counts = pd.crosstab(
    df_ext_final["grade_level"],
    df_ext_final["sex_group"],
    margins=True
)

display(ext_counts)

print("\nCurrent alcohol use proportion by grade and sex:")

ext_prop = (
    df_ext_final
    .groupby(["grade_level", "sex_group"])["current_alcohol_use"]
    .mean()
    .reset_index()
)

display(ext_prop)

Sample size by grade and sex:


sex_group,Female,Male,All
grade_level,,,
10th grade,1572,1550,3122
11th grade,1687,1477,3164
12th grade,1622,1611,3233
9th grade,1521,1560,3081
All,6402,6198,12600



Current alcohol use proportion by grade and sex:


,grade_level,sex_group,current_alcohol_use
0,10th grade,Female,0.421120
1,10th grade,Male,0.432258
2,11th grade,Female,0.444576
3,11th grade,Male,0.500339
4,12th grade,Female,0.509248
5,12th grade,Male,0.549969
6,9th grade,Female,0.403024
7,9th grade,Male,0.344872


## 10. Save cleaned extension data

In [16]:
ext_output_path = processed_dir / "cycle3_cleaned_grade_extension.csv"

df_ext_final.to_csv(ext_output_path, index=False)

print("Saved extension cleaned data to:", ext_output_path)

Saved extension cleaned data to: d:\作業\統計學實習\cycle3\project-cycle-3\data\processed\cycle3_cleaned_grade_extension.csv


## 11. Save variable notes

This creates a short markdown file documenting the variables and recoding rules.

In [17]:
variable_notes = '# Variable Notes for Cycle 3\n\n## Research Question\n\nIs the proportion of current alcohol use different between male and female students?\n\n## Main Variables\n\n### Group Variable: `WhatIsYourSex`\n\nOriginal coding used in this project:\n- `1 = Female`\n- `2 = Male`\n\nRecoded variable:\n- `sex_group = Female / Male`\n\n### Response Variable: `CurrentAlcoholUse`\n\nOriginal coding used in this project:\n- `1 = no current alcohol use`\n- `2–7 = current alcohol use`\n\nRecoded variable:\n- `current_alcohol_use = 0` means no current alcohol use\n- `current_alcohol_use = 1` means current alcohol use\n\n## Extension Variable: `InWhatGradeAreYou`\n\nUsed for descriptive grade-level comparison.\n\nCoding used in the extension:\n- `1 = 9th grade`\n- `2 = 10th grade`\n- `3 = 11th grade`\n- `4 = 12th grade`\n\nCode `5`, missing values, and invalid values are not used in the grade-level extension.\n\n## Cleaned Output Files\n\n- `data/processed/cycle3_cleaned_main.csv`\n- `data/processed/cycle3_cleaned_grade_extension.csv`\n'

notes_path = project_dir / "references" / "variable_notes.md"
notes_path.write_text(variable_notes, encoding="utf-8")

print("Saved variable notes to:", notes_path)

Saved variable notes to: d:\作業\統計學實習\cycle3\project-cycle-3\references\variable_notes.md


## 12. Final check

At the end of this notebook, you should have:

```text
data/processed/cycle3_cleaned_main.csv
data/processed/cycle3_cleaned_grade_extension.csv
references/variable_notes.md
```

Next notebook: `02_main_analysis.ipynb`.

In [18]:
print("Files created:")
for path in [main_output_path, ext_output_path, notes_path]:
    print(path, "->", path.exists())

Files created:
d:\作業\統計學實習\cycle3\project-cycle-3\data\processed\cycle3_cleaned_main.csv -> True
d:\作業\統計學實習\cycle3\project-cycle-3\data\processed\cycle3_cleaned_grade_extension.csv -> True
d:\作業\統計學實習\cycle3\project-cycle-3\references\variable_notes.md -> True
